In [1]:
from datasets import load_dataset

dataset=load_dataset("cornell-movie-review-data/rotten_tomatoes")

dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

In [2]:
def add_length(example):
    example["length"]=len(example["text"])
    return example
train_dataset=dataset["train"].map(add_length)

In [3]:
filtered=dataset["train"].filter(lambda x: x["label"]==2)
filtered

Dataset({
    features: ['text', 'label'],
    num_rows: 0
})

In [4]:
from transformers import AutoTokenizer

In [5]:
tokeinzer=AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [6]:
def tokeinze_function(example):
    return tokeinzer(example["text"],truncation=True)

tokeinze_function(dataset["train"][0])

{'input_ids': [101, 1996, 2600, 2003, 16036, 2000, 2022, 1996, 7398, 2301, 1005, 1055, 2047, 1000, 16608, 1000, 1998, 2008, 2002, 1005, 1055, 2183, 2000, 2191, 1037, 17624, 2130, 3618, 2084, 7779, 29058, 8625, 13327, 1010, 3744, 1011, 18856, 19513, 3158, 5477, 4168, 2030, 7112, 16562, 2140, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [7]:
tokeinzed_dataset=dataset.map(tokeinze_function)

In [8]:
tokeinzed_dataset=dataset.map(tokeinze_function,remove_columns=["text"])

In [9]:
print(len(tokeinzed_dataset["train"][0]["input_ids"]))
print(len(tokeinzed_dataset["train"][1]["input_ids"]))
print(len(tokeinzed_dataset["train"][2]["input_ids"]))


47
52
10


In [10]:
from transformers import DataCollatorWithPadding
data_collator=DataCollatorWithPadding(tokenizer=tokeinzer)

In [11]:
samples = dataset["train"][:2]
tokenized_samples=tokeinzer(samples["text"],truncation=True)

In [19]:
features=[
    {
        "input_ids":tokenized_samples["input_ids"][i],
        "token_type_ids":tokenized_samples["token_type_ids"][i],
        "attention_mask":tokenized_samples["attention_mask"][i]
    }
    for i in range(2)
]
print(len(features))

2


In [20]:
batch=data_collator(features)

print(batch["input_ids"].shape) 
print(batch["attention_mask"].shape) 


torch.Size([2, 52])
torch.Size([2, 52])
